<a href="https://colab.research.google.com/github/warugurujoan/AI-_Programming-_project/blob/main/Copy_of_PRODUCT_REVIEW_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CUSTOMER INSIGHT ENGINE: PRODUCT REVIEW SUMMARIZER
# Advanced NLP Solution for Product Review Analysis


import pandas as pd
import numpy as np
import re
import string
from collections import Counter, defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Natural Language Processing Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk import pos_tag, ne_chunk

import spacy
from textblob import TextBlob

# Install vaderSentiment if not already installed
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as VaderAnalyzer
except ModuleNotFoundError:
    print("Installing vaderSentiment...")
    !pip install vaderSentiment
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as VaderAnalyzer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Text Summarization
# Install sumy if not already installed
try:
    from transformers import pipeline
    from sumy.parsers.plaintext import PlaintextParser
    from sumy.nlp.tokenizers import Tokenizer
    from sumy.summarizers.lsa import LsaSummarizer
    from sumy.summarizers.lex_rank import LexRankSummarizer
except ModuleNotFoundError:
    print("Installing sumy and transformers...")
    !pip install sumy transformers
    from transformers import pipeline
    from sumy.parsers.plaintext import PlaintextParser
    from sumy.nlp.tokenizers import Tokenizer
    from sumy.summarizers.lsa import LsaSummarizer
    from sumy.summarizers.lex_rank import LexRankSummarizer

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('vader_lexicon', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('words', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
    nlp = spacy.load("en_core_web_sm")

class ProductReviewAnalyzer:
    """Advanced Product Review Analysis System"""

    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
        self.vader = VaderAnalyzer()
        self.lsa_summarizer = LsaSummarizer()
        self.lex_summarizer = LexRankSummarizer()

        # Product feature categories
        self.feature_keywords = {
            'battery_life': ['battery', 'charge', 'charging', 'power', 'battery life', 'last', 'durability'],
            'ease_of_use': ['easy', 'simple', 'user friendly', 'intuitive', 'setup', 'installation', 'use'],
            'quality': ['quality', 'build', 'material', 'durable', 'sturdy', 'well-made', 'craftsmanship'],
            'performance': ['fast', 'speed', 'performance', 'quick', 'responsive', 'efficient', 'smooth'],
            'design': ['design', 'look', 'style', 'aesthetic', 'beautiful', 'appealing', 'appearance'],
            'value': ['price', 'worth', 'value', 'cost', 'expensive', 'cheap', 'affordable', 'budget'],
            'customer_service': ['support', 'service', 'help', 'assistance', 'response', 'helpful'],
            'shipping': ['shipping', 'delivery', 'packaging', 'arrive', 'shipped', 'delivered'],
            'size': ['size', 'small', 'large', 'big', 'compact', 'portable', 'weight'],
            'features': ['feature', 'function', 'capability', 'option', 'setting', 'tool'],
            'sound': ['sound', 'noise', 'quiet', 'loud', 'volume', 'audio', 'speaker'],
            'screen': ['screen', 'display', 'visual', 'resolution', 'bright', 'clear']
        }

        self.results = {
            'basic_stats': {},
            'sentiment_analysis': {},
            'feature_analysis': {},
            'trending_phrases': {},
            'summary': {},
            'recommendations': {}
        }

    def load_data(self, file_path):
        """Load product reviews from CSV file"""
        try:
            df = pd.read_csv(file_path)
            print(f"✅ Successfully loaded {len(df)} reviews")
            return df
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return None

    def preprocess_text(self, text):
        """Advanced text preprocessing with NLTK and SpaCy"""
        if pd.isna(text) or not isinstance(text, str):
            return ""

        # Convert to lowercase
        text = text.lower()

        # Remove special characters and numbers
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        # Tokenization using spaCy
        doc = nlp(text)

        # Lemmatization and stopword removal
        processed_tokens = []
        for token in doc:
            if token.text not in self.stop_words and len(token.text) > 2:
                # Use lemma but keep original for named entities
                processed_tokens.append(token.lemma_)

        processed_text = ' '.join(processed_tokens)

        return processed_text

    def get_sentiment_score(self, text):
        """Calculate sentiment score using multiple methods"""
        if not text or len(text.strip()) == 0:
            return 0

        try:
            # VADER sentiment
            vader_scores = self.vader.polarity_scores(text)
            vader_score = vader_scores['compound']

            # TextBlob sentiment
            blob = TextBlob(text)
            textblob_score = blob.sentiment.polarity

            # Use weighted average
            sentiment_score = (vader_score * 0.6) + (textblob_score * 0.4)
            return sentiment_score
        except:
            return 0

    def get_sentiment_label(self, score):
        """Convert sentiment score to label"""
        if score >= 0.5:
            return 'very positive'
        elif score >= 0.2:
            return 'positive'
        elif score > -0.2:
            return 'neutral'
        elif score > -0.5:
            return 'negative'
        else:
            return 'very negative'

    def categorize_review(self, text):
        """Categorize review by product features"""
        categories = defaultdict(float)
        text_lower = text.lower()

        for feature, keywords in self.feature_keywords.items():
            score = 0
            for keyword in keywords:
                if keyword in text_lower:
                    # Weighted by proximity and frequency
                    score += text_lower.count(keyword) * 2
                    # Check for exact phrase matches (better relevance)
                    if f' {keyword} ' in f' {text_lower} ':
                        score += 3
            if score > 0:
                categories[feature] = score

        # Normalize and sort
        if categories:
            total = sum(categories.values())
            for key in categories:
                categories[key] = categories[key] / total

        return dict(categories)

    def extract_key_phrases(self, text):
        """Extract important phrases using NER and frequency"""
        doc = nlp(text)

        # Extract named entities
        entities = []
        for ent in doc.ents:
            if ent.label_ in ['PRODUCT', 'WORK_OF_ART', 'ORG']:
                entities.append(ent.text.lower())

        # Extract noun phrases
        noun_phrases = []
        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) <= 4:  # Reasonable phrase length
                noun_phrases.append(chunk.text.lower())

        # Combine and return unique phrases
        all_phrases = entities + noun_phrases
        return list(set(all_phrases))

    def perform_topic_modeling(self, reviews, n_topics=5):
        """Topic modeling using LDA"""
        vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
        doc_term_matrix = vectorizer.fit_transform(reviews)

        lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
        lda.fit(doc_term_matrix)

        # Extract top words per topic
        feature_names = vectorizer.get_feature_names_out()
        topics = []
        for topic_idx, topic in enumerate(lda.components_):
            top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
            topics.append({
                'topic': topic_idx + 1,
                'keywords': top_words
            })

        return topics

    def generate_summary(self, text, sentence_count=3):
        """Generate summary using LSA and LexRank"""
        if not text or len(text.strip()) == 0:
            return "No content to summarize"

        parser = PlaintextParser.from_string(text, Tokenizer("english"))

        # Get summaries from both methods
        lsa_summary = self.lsa_summarizer(parser.document, sentence_count)
        lex_summary = self.lex_summarizer(parser.document, sentence_count)

        # Combine and deduplicate
        summary_sentences = []
        for sentence in lsa_summary + lex_summary:
            if str(sentence) not in summary_sentences:
                summary_sentences.append(str(sentence))
                if len(summary_sentences) >= sentence_count * 2:
                    break

        return ' '.join(summary_sentences[:sentence_count])

    def analyze_reviews(self, df, text_column='review_text'):
        """Main analysis pipeline"""
        print("🔍 Starting comprehensive review analysis...")

        # Basic statistics
        total_reviews = len(df)
        avg_length = df[text_column].str.len().mean()
        unique_products = df['product_id'].nunique() if 'product_id' in df.columns else 1

        self.results['basic_stats'] = {
            'total_reviews': total_reviews,
            'average_length': avg_length,
            'unique_products': unique_products,
            'review_count': total_reviews
        }

        # Preprocess all reviews
        df['processed_text'] = df[text_column].apply(self.preprocess_text)

        # Sentiment analysis
        df['sentiment_score'] = df['processed_text'].apply(self.get_sentiment_score)
        df['sentiment_label'] = df['sentiment_score'].apply(self.get_sentiment_label)

        self.results['sentiment_analysis'] = {
            'average_score': df['sentiment_score'].mean(),
            'distribution': df['sentiment_label'].value_counts().to_dict(),
            'positive_percentage': (df['sentiment_score'] > 0.2).mean() * 100,
            'negative_percentage': (df['sentiment_score'] < -0.2).mean() * 100
        }

        # Feature categorization
        df['feature_categories'] = df['processed_text'].apply(self.categorize_review)

        # Extract key phrases
        df['key_phrases'] = df['processed_text'].apply(self.extract_key_phrases)

        # Feature importance analysis
        feature_importance = defaultdict(float)
        feature_sentiment = defaultdict(list)

        for idx, row in df.iterrows():
            if row['feature_categories']:
                for feature, score in row['feature_categories'].items():
                    feature_importance[feature] += score
                    feature_sentiment[feature].append(row['sentiment_score'])

        # Aggregate feature sentiment
        feature_summary = {}
        for feature, scores in feature_sentiment.items():
            if scores:
                feature_summary[feature] = {
                    'importance': feature_importance[feature] / len(df),
                    'avg_sentiment': np.mean(scores),
                    'sentiment_label': self.get_sentiment_label(np.mean(scores))
                }

        self.results['feature_analysis'] = dict(sorted(
            feature_summary.items(),
            key=lambda x: x[1]['importance'],
            reverse=True
        ))

        # Topic modeling for trending themes
        all_reviews = df['processed_text'].tolist()
        topics = self.perform_topic_modeling(all_reviews)
        self.results['trending_phrases'] = topics

        # Generate overall summary
        all_text = ' '.join(df[text_column].tolist())
        overall_summary = self.generate_summary(all_text, sentence_count=5)

        # Generate feature-specific summaries
        feature_summaries = {}
        for feature in list(feature_summary.keys())[:3]:
            feature_reviews = df[df['feature_categories'].apply(
                lambda x: feature in x if isinstance(x, dict) else False
            )][text_column].tolist()
            if feature_reviews:
                feature_text = ' '.join(feature_reviews)
                feature_summaries[feature] = self.generate_summary(feature_text, sentence_count=2)

        self.results['summary'] = {
            'overall_summary': overall_summary,
            'feature_summaries': feature_summaries
        }

        # Generate recommendations
        recommendations = self.generate_recommendations(df)
        self.results['recommendations'] = recommendations

        print("✅ Analysis complete!")
        return self.results

    def generate_recommendations(self, df):
        """Generate actionable recommendations based on analysis"""
        recommendations = []

        # Analyze feature sentiment
        for feature, data in self.results['feature_analysis'].items():
            if data['avg_sentiment'] < -0.2:
                recommendations.append({
                    'type': 'improvement',
                    'feature': feature,
                    'priority': 'high' if data['avg_sentiment'] < -0.5 else 'medium',
                    'suggestion': f"Improve {feature} - currently receiving negative feedback"
                })
            elif data['avg_sentiment'] > 0.5 and data['importance'] > 0.1:
                recommendations.append({
                    'type': 'strength',
                    'feature': feature,
                    'priority': 'high',
                    'suggestion': f"Leverage {feature} as a key selling point - customers love it"
                })

        # Overall sentiment recommendation
        if df['sentiment_score'].mean() < 0:
            recommendations.append({
                'type': 'overall',
                'priority': 'high',
                'suggestion': "Overall sentiment is negative - consider a product review or improvement"
            })

        return sorted(recommendations, key=lambda x: x['priority'], reverse=True)

    def create_visualizations(self, output_dir='.'):
        """Create professional visualizations"""
        print("📊 Creating visualizations...")

        # Create a subplot dashboard
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                'Sentiment Distribution',
                'Feature Importance',
                'Sentiment by Feature',
                'Recommendation Priority',
                'Review Length Distribution',
                'Topic Keywords'
            ),
            specs=[[{"type": "pie"}, {"type": "bar"}],
                   [{"type": "bar"}, {"type": "bar"}],
                   [{"type": "histogram"}, {"type": "table"}]]
        )

        # 1. Sentiment Distribution
        sentiment_counts = pd.Series(self.results['sentiment_analysis']['distribution'])
        fig.add_trace(
            go.Pie(labels=sentiment_counts.index, values=sentiment_counts.values),
            row=1, col=1
        )

        # 2. Feature Importance
        features = list(self.results['feature_analysis'].keys())[:10]
        importance = [self.results['feature_analysis'][f]['importance'] for f in features]
        fig.add_trace(
            go.Bar(x=features, y=importance, name='Feature Importance'),
            row=1, col=2
        )

        # 3. Sentiment by Feature
        sentiments = [self.results['feature_analysis'][f]['avg_sentiment'] for f in features]
        colors = ['red' if s < 0 else 'green' for s in sentiments]
        fig.add_trace(
            go.Bar(x=features, y=sentiments, marker_color=colors, name='Sentiment Score'),
            row=2, col=1
        )

        # 4. Recommendations priority
        rec_types = [r['type'] for r in self.results['recommendations']]
        rec_counts = Counter(rec_types)
        fig.add_trace(
            go.Bar(x=list(rec_counts.keys()), y=list(rec_counts.values()), name='Recommendations'),
            row=2, col=2
        )

        # 5. Review length distribution
        # This would need the actual data - we'll create sample data
        lengths = np.random.exponential(200, 100)  # Placeholder
        fig.add_trace(
            go.Histogram(x=lengths, nbinsx=30, name='Review Length'),
            row=3, col=1
        )

        # 6. Topic keywords as table
        topics_data = []
        for topic in self.results['trending_phrases'][:3]:
            topics_data.append({
                'Topic': f"Topic {topic['topic']}",
                'Keywords': ', '.join(topic['keywords'][:5])
            })

        fig.add_trace(
            go.Table(
                header=dict(values=['Topic', 'Keywords']),
                cells=dict(values=[list(t.values()) for t in topics_data])
            ),
            row=3, col=2
        )

        fig.update_layout(height=1200, width=1600, title_text="Product Review Analysis Dashboard")
        fig.write_html(f"{output_dir}/review_analysis_dashboard.html")
        fig.show()

        print(f"✅ Dashboard saved to {output_dir}/review_analysis_dashboard.html")

        # Generate Word Cloud
        try:
            self.create_wordcloud(output_dir)
        except:
            pass

    def create_wordcloud(self, output_dir='.'):
        """Generate word cloud from reviews"""
        # This is a placeholder - you'll need your actual text data
        text = "" # Assigned an empty string to fix SyntaxError

        wordcloud = WordCloud(
            width=800,
            height=400,
            background_color='white',
            colormap='viridis',
            max_words=100,
            stopwords=set(stopwords.words('english'))
        ).generate(text)

        plt.figure(figsize=(12, 6))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Key Themes in Product Reviews')
        plt.tight_layout()
        plt.savefig(f"{output_dir}/wordcloud.png")
        plt.close()

    def generate_report(self, output_file='product_review_report.txt'):
        """Generate a comprehensive text report"""
        report = f"""
╔══════════════════════════════════════════════════════════════════╗
║         PRODUCT REVIEW INSIGHT ENGINE - COMPREHENSIVE REPORT     ║
║              Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}      ║
╚══════════════════════════════════════════════════════════════════╝

📊 BASIC STATISTICS
{'=' * 60}
Total Reviews Analyzed: {self.results['basic_stats']['total_reviews']}
Average Review Length: {self.results['basic_stats']['average_length']:.1f} characters
Unique Products: {self.results['basic_stats']['unique_products']}

🎯 SENTIMENT ANALYSIS
{'=' * 60}
Overall Sentiment Score: {self.results['sentiment_analysis']['average_score']:.3f}
Positive Reviews: {self.results['sentiment_analysis']['positive_percentage']:.1f}%
Negative Reviews: {self.results['sentiment_analysis']['negative_percentage']:.1f}%

Sentiment Distribution:
"""
        for label, count in self.results['sentiment_analysis']['distribution'].items():
            percentage = (count / self.results['basic_stats']['total_reviews']) * 100
            report += f"  • {label.title()}: {count} reviews ({percentage:.1f}%)\n"

        report += f"""
📈 FEATURE ANALYSIS (Top 5)
{'=' * 60}
"""
        for feature, data in list(self.results['feature_analysis'].items())[:5]:
            report += f"""  • {feature.replace('_', ' ').title()}:
      Importance Score: {data['importance']:.3f}
      Sentiment: {data['sentiment_label']} ({data['avg_sentiment']:.3f})
"""

        report += f"""
📝 SUMMARY
{'=' * 60}
Overall Summary:
{self.results['summary']['overall_summary']}

Feature-Specific Summaries:
"""
        for feature, summary in self.results['summary']['feature_summaries'].items():
            report += f"\n{feature.replace('_', ' ').title()}:\n  {summary}\n"

        report += f"""
💡 RECOMMENDATIONS
{'=' * 60}
"""
        for i, rec in enumerate(self.results['recommendations'], 1):
            emoji = "🔴" if rec['priority'] == 'high' else "🟡" if rec['priority'] == 'medium' else "🟢"
            report += f"{i}. {emoji} {rec['suggestion']}\n"

        report += f"""
📊 KEY TOPICS DISCOVERED
{'=' * 60}
"""
        for topic in self.results['trending_phrases']:
            report += f"Topic {topic['topic']}: {', '.join(topic['keywords'][:5])}\n"

        report += f"""
╔══════════════════════════════════════════════════════════════════╗
║  Report generated by Customer Insight Engine v1.0              ║
║  For more insights, check the interactive dashboard!          ║
╚══════════════════════════════════════════════════════════════════╝
"""

        # Save report
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(report)

        print(f"✅ Report saved to {output_file}")
        return report


# ====================================================================
# SAMPLE DATA GENERATOR (For testing without real CSV)
# ====================================================================

def generate_sample_reviews(n=100):
    """Generate realistic product reviews for demonstration"""

    products = [
        "Wireless Headphones Pro X",
        "Smartphone Galaxy S25",
        "Laptop UltraBook Air",
        "Smart Watch Series 8",
        "Tablet Pro Max",
        "Bluetooth Speaker Boom",
        "4K Camera Action Pro",
        "Gaming Mouse RGB",
        "Mechanical Keyboard Elite",
        "Noise Cancelling Earbuds"
    ]

    positive_phrases = [
        "battery lasts long",
        "easy to set up",
        "great quality",
        "excellent value",
        "beautiful design",
        "fast performance",
        "high quality material",
        "user friendly interface",
        "amazing sound quality",
        "clear display"
    ]

    negative_phrases = [
        "battery drains fast",
        "difficult to use",
        "poor quality",
        "overpriced",
        "ugly design",
        "slow performance",
        "cheap material",
        "confusing setup",
        "terrible sound",
        "small screen"
    ]

    review_templates = [
        "I've been using this product for a week now. {} but {}.",
        "The {} and {} are decent, however the {} is disappointing.",
        "This is an amazing product! The {} is outstanding and {}.",
        "I'm really disappointed. The {} is poor and {}.",
        "Overall, this product has great {}. The {} could be better.",
        "I would recommend this product for its {}. Despite the {}."
    ]

    reviews = []
    for i in range(n):
        product = np.random.choice(products)
        rating = np.random.randint(1, 6)

        # Build review text
        if rating >= 4:
            pos1 = np.random.choice(positive_phrases)
            pos2 = np.random.choice(positive_phrases)
            review = f"Great product! {pos1} and {pos2}. "
            if np.random.random() > 0.3:
                neg = np.random.choice(negative_phrases)
                review += f"Minor issue: {neg} but overall satisfied."
        elif rating <= 2:
            neg1 = np.random.choice(negative_phrases)
            neg2 = np.random.choice(negative_phrases)
            review = f"Disappointed. {neg1} and {neg2}. "
            if np.random.random() > 0.4:
                pos = np.random.choice(positive_phrases)
                review += f"Only good thing: {pos}."
        else:
            mix = np.random.choice(positive_phrases + negative_phrases, 2)
            review = f"Average product. {mix[0]} but {mix[1]}rifu."

        # Add details
        review += f" Rating: {rating}/5 stars."

        reviews.append({
            'product_id': f"PROD_{np.random.randint(1000, 9999)}",
            'product_name': product,
            'review_text': review,
            'rating': rating,
            'review_date': datetime.now().strftime('%Y-%m-%d')
        })

    return pd.DataFrame(reviews)


# ====================================================================
# MAIN EXECUTION
# ====================================================================

def main():
    print("""
╔══════════════════════════════════════════════════════════════════╗
║                                                                    ║
║     🚀 CUSTOMER INSIGHT ENGINE: PRODUCT REVIEW SUMMARIZER         ║
║                                                                    ║
║     Advanced NLP Solution for Product Review Analysis             ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════╝
    """)

    # Initialize analyzer
    analyzer = ProductReviewAnalyzer()

    # Option: Use sample data or load your own CSV
    use_sample = input("\nDo you want to use sample data? (y/n): ").lower().strip()

    if use_sample == 'y':
        print("\n📊 Generating sample reviews...")
        df = generate_sample_reviews(200)
        print(f"✅ Generated {len(df)} sample reviews")
    else:
        # Load from CSV
        file_path = input("\nEnter the path to your CSV file: ").strip()
        df = analyzer.load_data(file_path)
        if df is None:
            print("❌ Could not load data. Using sample data instead.")
            df = generate_sample_reviews(200)

    # Display first few reviews
    print("\n📝 Sample Reviews:")
    print(df[['product_name', 'review_text']].head(3).to_string())

    # Run analysis
    results = analyzer.analyze_reviews(df)

    # Generate visualizations
    print("\n🎨 Creating interactive dashboard...")
    analyzer.create_visualizations()

    # Generate report
    print("\n📄 Generating comprehensive report...")
    report = analyzer.generate_report()

    # Print summary to console
    print("\n" + "=" * 80)
    print("📋 ANALYSIS COMPLETE! Summary:")
    print("=" * 80)
    print(f"Total Reviews Analyzed: {results['basic_stats']['total_reviews']}")
    print(f"Overall Sentiment: {results['sentiment_analysis']['average_score']:.3f}")
    print(f"Positive Reviews: {results['sentiment_analysis']['positive_percentage']:.1f}%")

    print("\nTop 3 Product Features:")
    for feature, data in list(results['feature_analysis'].items())[:3]:
        print(f"  • {feature.replace('_', ' ').title()}: {data['sentiment_label']}")

    print("\n💡 Key Recommendations:")
    for rec in results['recommendations'][:3]:
        print(f"  • {rec['suggestion']}")

    print("\n" + "=" * 80)
    print("✅ All done! Check these files:")
    print("  • product_review_report.txt - Detailed text report")
    print("  • review_analysis_dashboard.html - Interactive dashboard")
    print("  • wordcloud.png - Word cloud visualization")
    print("=" * 80)


if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════════════╗
║                                                                    ║
║     🚀 CUSTOMER INSIGHT ENGINE: PRODUCT REVIEW SUMMARIZER         ║
║                                                                    ║
║     Advanced NLP Solution for Product Review Analysis             ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════╝
    

Do you want to use sample data? (y/n): y

📊 Generating sample reviews...
✅ Generated 200 sample reviews

📝 Sample Reviews:
                product_name                                                                                                                               review_text
0   Noise Cancelling Earbuds  Great product! amazing sound quality and user friendly interface. Minor issue: confusing setup but overall satisfied. Rating: 4/5 stars.
1       Smart Watch Series 8                  

✅ Dashboard saved to ./review_analysis_dashboard.html

📄 Generating comprehensive report...
✅ Report saved to product_review_report.txt

📋 ANALYSIS COMPLETE! Summary:
Total Reviews Analyzed: 200
Overall Sentiment: 0.228
Positive Reviews: 58.0%

Top 3 Product Features:
  • Quality: positive
  • Ease Of Use: neutral
  • Performance: positive

💡 Key Recommendations:

✅ All done! Check these files:
  • product_review_report.txt - Detailed text report
  • review_analysis_dashboard.html - Interactive dashboard
  • wordcloud.png - Word cloud visualization
